# Pipeline Definitivo — IMPACTO_FRAUDE (multiclase)

Pipeline completo para clasificación multiclase del impacto.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score)
import xgboost as xgb
import lightgbm as lgb
try:
    import catboost as cb
    CATBOOST_AVAIL = True
except ImportError:
    CATBOOST_AVAIL = False
    print('CatBoost no instalado — se omite')
sys.path.append(str(Path.cwd().parent))
from model.feature_engineering import *
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('OK')

## Pipeline completo

In [ ]:
def train_pipeline_multiclass(df, target='IMPACTO_FRAUDE', test_size=0.2, random_state=42):
    # Drop del otro target para evitar data leakage
    other = [t for t in ['IS_FRAUD', 'IMPACTO_FRAUDE'] if t != target]
    df = df.drop(columns=other, errors='ignore')
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target])
    print(f'Train: {train_df.shape[0]}  Test: {test_df.shape[0]}')
    
    # KNN Imputation
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    imputer = KNNImputer(n_neighbors=3)
    train_df[num_cols] = imputer.fit_transform(train_df[num_cols])
    test_df[num_cols] = imputer.transform(test_df[num_cols])
    
    # Feature Engineering
    fe = FeatureEngineer(encode_target=target, random_state=random_state)
    X_train = fe.fit_transform(train_df); y_train = X_train.pop(target).values
    X_test = fe.transform(test_df); y_test = X_test.pop(target).values
    num_feats = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    Xtr = X_train.copy(); Xte = X_test.copy()
    Xtr[num_feats] = scaler.fit_transform(X_train[num_feats])
    Xte[num_feats] = scaler.transform(X_test[num_feats])
    print(f'Features: {Xtr.shape[1]}')

    models = [
        ('LogisticRegression', LogisticRegression(solver='lbfgs', max_iter=1000), {'C': [0.1, 1, 10]}),
        ('RandomForest', RandomForestClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
        ('XGBoost', xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='mlogloss'), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
        ('LightGBM', lgb.LGBMClassifier(random_state=42, verbose=-1), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ]
    if CATBOOST_AVAIL:
        models.append(('CatBoost', cb.CatBoostClassifier(random_state=42, verbose=False, loss_function='MultiClass'),
                       {'iterations': [100, 200], 'depth': [4, 6], 'learning_rate': [0.05, 0.1]}))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []; bests = {}
    for name, est, params in models:
        print(f'\n>>> {name}')
        gs = GridSearchCV(est, params, cv=cv, scoring='f1_weighted', n_jobs=-1)
        gs.fit(Xtr, y_train)
        bests[name] = gs.best_estimator_
        yp = gs.predict(Xte)
        results.append({'modelo': name, 'cv_f1': gs.best_score_, 'test_f1': f1_score(y_test, yp, average='weighted')})
        print(f'  CV={gs.best_score_:.4f} Test F1={results[-1]["test_f1"]:.4f}')
    
    results_df = pd.DataFrame(results).sort_values('test_f1', ascending=False)
    best_model = bests[results_df.iloc[0]['modelo']]
    print(f'\nMejor: {results_df.iloc[0]["modelo"]}')
    return best_model, fe, scaler, imputer, results_df, Xtr, Xte, y_train, y_test

DATA_PATH = Path.cwd().parent / 'Notebooks' / 'data' / 'dataset_fraude.csv'
df = pd.read_csv(DATA_PATH)
best_model, fe, scaler, imputer, res, Xtr, Xte, ytr, yte = train_pipeline_multiclass(df, target='IMPACTO_FRAUDE')
print('\n' + '='*50)
print(res.round(4).to_string(index=False))

## 2. Evaluación

In [ ]:
best_name = res.iloc[0]['modelo']
print(f'=== {best_name} ===')
yp = best_model.predict(Xte)
print(classification_report(yte, yp, digits=4))

In [ ]:
cm = confusion_matrix(yte, yp)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}'); plt.tight_layout(); plt.show()

## 3. Guardar

In [ ]:
import joblib
MODEL_DIR = Path.cwd().parent / 'model' / 'saved_models'
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(best_model, MODEL_DIR / 'best_model_impacto.pkl')
joblib.dump(fe, MODEL_DIR / 'fe_impacto.pkl')
joblib.dump(scaler, MODEL_DIR / 'scaler_impacto.pkl')
joblib.dump(imputer, MODEL_DIR / 'imputer_impacto.pkl')
print('Guardado')